In [ ]:
import boto3

# Load the JSON data from the file
MODEL_URL = "https://s3.waw3-1.cloudferro.com/swift/v1/ecdc-waw3-1-ekqouvq3otv8hmw0njzuvo0g4dy0ys8r985n7dggjis3erkpn5o/models/"

# Replace with your CloudFerro S3 endpoint and credentials
s3_endpoint = 'https://s3.waw3-1.cloudferro.com'
access_key = "xx"
secret_key = "yy"
bucket_name = 'ecdc-waw3-1-ekqouvq3otv8hmw0njzuvo0g4dy0ys8r985n7dggjis3erkpn5o' 
s3_directory = 'models'  # Path in the S3 bucket

session = boto3.session.Session()
s3_client = session.client('s3',
                            endpoint_url=s3_endpoint,
                            aws_access_key_id=access_key,
                            aws_secret_access_key=secret_key)

# List objects in the specified directory
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=s3_directory)

# Filter and create a list of .onnx file names
onnx_files = []
if 'Contents' in response:
    model_urls = [MODEL_URL+ obj['Key'].split('/')[-1] for obj in response['Contents'] if obj['Key'].endswith('.onnx')]



In [9]:
import sys
import os
import pystac
from datetime import datetime

sys.path.append(os.path.abspath('C:/Git_projects/eo_processing/src'))

from eo_processing.utils.onnx_model_utilities import get_training_features_from_model

def create_stac_item(model_url):
    # Get the training features (input and output features) for each model
    metadata = get_training_features_from_model(model_url)
    
    # Assuming metadata has the format you're working with:
    input_features = metadata['input_features']
    output_features = metadata['output_features']
    
    # Create the STAC Item metadata
    item = pystac.Item(
        id=model_url.split("/")[-1].split('.')[0],  # Unique ID based on the model name
        geometry=None,  # Add geometry if available, otherwise leave it as None
        bbox=None,  # You can provide a bounding box if available
        datetime=datetime.utcnow(),  # Current datetime for the item
        properties={
            'model_url': model_url,
            'input_features': input_features,
            'output_features': output_features,
            'model_format': 'ONNX',  # Specifying the model format
            'spatial_extent': [-30.0, 30.0, 60.0, 80.0],
            'temporal_extent': [(datetime(2020, 1, 1).isoformat(), datetime(2024, 12, 31).isoformat())]
        },
        assets={
            'model': pystac.Asset(
                href=model_url,
                media_type='application/onnx',  # Specifying the media type
            )
        },
    )

    return item

# Iterate through the model URLs and create the STAC items
stac_items = [create_stac_item(url) for url in model_urls]



Loaded ONNX model from /tmp/cache\Level1_class-0_129predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-C_71predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-D_68predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-E_85predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-F_90predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-G_164predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-H_65predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-I_50predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-J_62predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level2_class-X_54predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level3_class-C1_62predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level3_class-C3_62predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level3_class-E1_50predictors_v1.onnx
Loaded ONNX model from /tmp/cache\Level3_class-E2_50predictors_v1.onnx
Loaded ONNX mo

In [10]:
# Create a STAC Catalog using pystac (without passing links in constructor)
catalog = pystac.Catalog(
    id='model-catalog',
    description='Catalog of machine learning models'
)

# Add a self link to the catalog using add_link
catalog.add_link(pystac.Link(rel='self', target='catalog.json'))  # Use target instead of href

# Add items to the catalog
for item in stac_items:
    catalog.add_item(item)

# Save the catalog to a local file
catalog.save('model_catalog.json')

print("STAC catalog created and saved as 'model_catalog.json'.")

STAC catalog created and saved as 'model_catalog.json'.
